# Getting Started with Cost and Performance Optimization in Snowflake

**Source:** [Snowflake Quickstart Guide](https://www.snowflake.com/en/developers/guides/getting-started-cost-performance-optimization/)

## Overview

This notebook walks you through various optimization features available in Snowflake to help you manage costs and improve query performance. By completing this guide, you will understand and implement:

1. **Setup Environment** — Create the required roles, warehouses, and sample datasets
2. **Warehouse Controls** — Leverage settings on a virtual warehouse to optimize usage
3. **Account Usage** — Use the Account Usage schema to uncover savings opportunities
4. **Storage Monitoring** — Identify cost savings with high-churn and short-lived tables
5. **Automatic Clustering** — Reduce micro-partition scans for faster queries
6. **Materialized Views** — Speed up expensive aggregations with pre-computed datasets
7. **Query Acceleration** — Offload outlier queries to shared compute resources
8. **Search Optimization** — Improve performance of selective point lookups

### Prerequisites

- Familiarity with the [Snowflake platform](https://docs.snowflake.com/en/user-guide/intro-key-concepts)
- Basic understanding of [micro-partitions](https://docs.snowflake.com/en/user-guide/tables-clustering-micropartitions)
- [ACCOUNTADMIN](https://docs.snowflake.com/en/user-guide/security-access-control-considerations#using-the-accountadmin-role) access on a Snowflake account

---
## 1. Environment Setup

This section creates the necessary infrastructure for the hands-on lab:

- **Warehouse:** A SMALL standard warehouse (`hol_compute_wh`) that starts suspended and does not auto-resume — giving you full control over when credits are consumed.
- **Database & Schema:** `OPT_HOL.DEMO` to hold all lab tables.
- **Tables:** Copies of TPC-H SF100 data (`lineitem`, `orders`, `part`) to demonstrate optimization features.
- **Clustered Table:** `lineitem_cl` with a `LINEAR(l_shipdate)` clustering key to compare clustered vs. unclustered performance.
- **Materialized View:** `lineitem_mv` to demonstrate automatic query rewriting.
- **Additional Tables:** `DATE_DIM` and `CATALOG_RETURNS` from TPC-DS for Search Optimization demos.

> **Note:** The `lineitem` table from TPC-H SF100 contains ~600M rows. The CTAS operations below may take a few minutes on a SMALL warehouse.

In [ ]:
-- Set role and create a dedicated warehouse for this lab
USE ROLE ACCOUNTADMIN;

CREATE WAREHOUSE IF NOT EXISTS hol_compute_wh
WITH WAREHOUSE_SIZE = 'SMALL'
     WAREHOUSE_TYPE = 'STANDARD'
     INITIALLY_SUSPENDED = TRUE
     AUTO_RESUME = FALSE;

USE WAREHOUSE hol_compute_wh;

In [ ]:
-- Create the database and schema for the lab
CREATE DATABASE IF NOT EXISTS OPT_HOL;
USE DATABASE OPT_HOL;
CREATE SCHEMA IF NOT EXISTS DEMO;
USE SCHEMA OPT_HOL.DEMO;

In [ ]:
-- Create base tables from TPC-H SF100 sample data
-- lineitem is ordered by L_PARTKEY intentionally (NOT by shipdate) to demonstrate clustering benefits later
CREATE OR REPLACE TABLE lineitem AS
  SELECT * FROM snowflake_sample_data.tpch_sf100.lineitem ORDER BY L_PARTKEY;

CREATE OR REPLACE TABLE orders AS
  SELECT * FROM snowflake_sample_data.tpch_sf100.orders;

CREATE OR REPLACE TABLE part AS
  SELECT * FROM snowflake_sample_data.tpch_sf100.part;

In [ ]:
-- Create a clustered copy of lineitem with LINEAR clustering on l_shipdate
-- This allows Snowflake's Automatic Clustering service to co-locate rows by shipdate
CREATE OR REPLACE TABLE lineitem_cl AS SELECT * FROM lineitem;
ALTER TABLE lineitem_cl CLUSTER BY LINEAR(l_shipdate);

In [ ]:
-- Create a Materialized View on the clustered table
-- Snowflake can automatically rewrite user queries to use this MV (query rewrite)
CREATE OR REPLACE MATERIALIZED VIEW lineitem_mv AS
SELECT
    TO_CHAR(l_shipdate, 'YYYYMM') AS ship_month,
    l_orderkey,
    SUM(l_quantity * l_extendedprice) AS order_price,
    SUM(l_quantity * l_discount) AS order_discount,
    order_price - order_discount AS net_selling_price
FROM
    lineitem_cl
GROUP BY
    TO_CHAR(l_shipdate, 'YYYYMM'),
    l_orderkey;

In [ ]:
-- Create tables from TPC-DS for the Search Optimization section
CREATE OR REPLACE TABLE DATE_DIM AS
  SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.DATE_DIM;

CREATE OR REPLACE TABLE CATALOG_RETURNS AS
  SELECT cr.*
  FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.CATALOG_RETURNS cr
  JOIN SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.DATE_DIM d
    ON cr.cr_returned_date_sk = d.d_date_sk
  WHERE d.d_year IN (2001, 2002);

---
## 2. Warehouse Controls

Virtual warehouses are the compute engines in Snowflake. Properly configuring warehouse settings is one of the simplest and most impactful ways to control costs:

| Setting | Purpose |
|---------|---------|
| **AUTO_SUSPEND** | Automatically suspends the warehouse after N seconds of inactivity (saves credits) |
| **AUTO_RESUME** | Automatically resumes the warehouse when a query is submitted |
| **STATEMENT_TIMEOUT** | Kills long-running queries after a threshold (prevents runaway costs) |
| **RESOURCE_MONITOR** | Sets credit quotas with notify/suspend triggers to prevent overspend |

### Best Practices
- Set `AUTO_SUSPEND` to 60 seconds for interactive/ad-hoc warehouses.
- Always enable `AUTO_RESUME` so users don't have to manually start warehouses.
- Use statement timeouts to catch accidental cartesian joins or infinite loops.
- Assign resource monitors to all warehouses to enforce credit budgets.

In [ ]:
-- View current parameters/settings for the warehouse
USE ROLE ACCOUNTADMIN;
SHOW PARAMETERS FOR WAREHOUSE hol_compute_wh;

In [ ]:
-- Configure auto-suspend (60 seconds) and auto-resume
ALTER WAREHOUSE hol_compute_wh SET AUTO_SUSPEND = 60;
ALTER WAREHOUSE hol_compute_wh SET AUTO_RESUME = TRUE;

In [ ]:
-- Set statement timeout: account-level (2 hours) and warehouse-level (1 hour)
-- Warehouse-level overrides account-level for queries running on that warehouse
ALTER ACCOUNT SET STATEMENT_TIMEOUT_IN_SECONDS = 7200;
ALTER WAREHOUSE hol_compute_wh SET STATEMENT_TIMEOUT_IN_SECONDS = 3600;

In [ ]:
-- Create a Resource Monitor with credit quota and trigger actions
-- NOTIFY at 75%, SUSPEND new queries at 100%, SUSPEND_IMMEDIATE (kill running) at 110%
CREATE OR REPLACE RESOURCE MONITOR Credits_Quota_Monitoring
  WITH CREDIT_QUOTA = 5000
       NOTIFY_USERS = (JDOE, "Jane Smith", "John Doe")
  TRIGGERS ON 75 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND
           ON 110 PERCENT DO SUSPEND_IMMEDIATE;

-- Assign the resource monitor to our warehouse
ALTER WAREHOUSE hol_compute_wh SET RESOURCE_MONITOR = Credits_Quota_Monitoring;

---
## 3. Account Usage Queries

[Account Usage](https://docs.snowflake.com/en/sql-reference/account-usage) is a powerful tool in an administrator's toolbox to identify optimization opportunities. The `SNOWFLAKE.ACCOUNT_USAGE` schema contains:

- **Metadata** about all objects in the account
- **Credit consumption** metrics (warehouse, serverless services, pipes)
- **Storage usage** metrics
- **Data transfer** metrics
- **Query History** — full detail on every query: execution plan, rows scanned, warehouse used, query text, etc.

> **Note:** Account Usage views have a latency of 45 minutes to 3 hours. For real-time data, use `INFORMATION_SCHEMA` (limited to 7 days of history).

In [ ]:
-- Set context for Account Usage queries
USE ROLE ACCOUNTADMIN;
USE SCHEMA SNOWFLAKE.ACCOUNT_USAGE;
USE WAREHOUSE hol_compute_wh;

In [ ]:
-- Warehouse credit consumption over time
SELECT * FROM snowflake.account_usage.warehouse_metering_history LIMIT 10;

In [ ]:
-- Access History: shows which objects were read/written by queries (useful for lineage & governance)
SELECT * FROM snowflake.account_usage.access_history LIMIT 10;

In [ ]:
-- Snowpipe (continuous ingestion) usage metrics
SELECT * FROM snowflake.account_usage.pipe_usage_history LIMIT 10;

In [ ]:
-- Overall storage usage (stage, database, failsafe)
SELECT * FROM snowflake.account_usage.storage_usage LIMIT 10;

In [ ]:
-- Detailed per-table storage breakdown (active, time-travel, failsafe, clone bytes)
SELECT * FROM snowflake.account_usage.table_storage_metrics LIMIT 10;

---
## 4. Storage Usage Monitoring

Storage costs in Snowflake come from three sources beyond active data:
1. **Time Travel** — retained historical data for `UNDROP` and `AT/BEFORE` queries
2. **Fail-safe** — 7 days of disaster recovery (non-queryable, after time travel expires)
3. **Clone retention** — bytes retained for zero-copy clones

### Key Patterns to Identify
- **High-churn tables** — tables with frequent DML that accumulate large time-travel/failsafe bytes relative to active data
- **Short-lived tables** — tables that are created and dropped within 24 hours (ETL staging)
- **Unused tables** — tables not accessed in 90+ days

### Recommended Actions
| Pattern | Action |
|---------|--------|
| High churn (>40% non-active bytes) | Reduce `DATA_RETENTION_TIME_IN_DAYS` or use `TRANSIENT` tables |
| Short-lived (<24h) | Use `TRANSIENT` or `TEMPORARY` table types (no failsafe) |
| Unused (>90 days) | Investigate business value; consider archiving or dropping |

In [ ]:
-- Identify high-churn tables (non-active bytes > 40% of active) and short-lived tables (< 24 hours)
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE hol_compute_wh;

SELECT
    t.table_catalog || '.' || t.table_schema || '.' || t.table_name AS fq_table_name,
    t.active_bytes / POWER(1024, 3) AS active_size_gb,
    t.time_travel_bytes / POWER(1024, 3) AS time_travel_gb,
    t.failsafe_bytes / POWER(1024, 3) AS failsafe_gb,
    t.retained_for_clone_bytes / POWER(1024, 3) AS clone_retain_gb,
    active_size_gb + time_travel_gb + failsafe_gb + clone_retain_gb AS total_size_gb,
    (t.time_travel_bytes + t.failsafe_bytes + t.retained_for_clone_bytes) / POWER(1024, 3) AS non_active_size_gb,
    DIV0(non_active_size_gb, active_size_gb) * 100 AS churn_pct,
    t.deleted,
    TIMEDIFF('hour', t.table_created, t.table_dropped) AS table_life_duration_hours,
    t1.is_transient,
    t1.table_type,
    t1.retention_time,
    t1.auto_clustering_on,
    t1.clustering_key,
    t1.last_altered,
    t1.last_ddl
FROM
    snowflake.account_usage.table_storage_metrics t
    JOIN snowflake.account_usage.tables t1
      ON t.id = t1.table_id
WHERE
    1 = 1
    -- AND t1.table_catalog IN ('','') -- uncomment to filter specific databases
    AND (
        churn_pct >= 40
        OR table_life_duration_hours <= 24  -- short-lived tables
    )
ORDER BY total_size_gb DESC;

In [ ]:
-- Identify tables not altered in the past 90 days (potential candidates for archival/removal)
SELECT
    TABLE_CATALOG || '.' || TABLE_SCHEMA || '.' || TABLE_NAME AS TABLE_PATH,
    TABLE_NAME,
    TABLE_SCHEMA AS SCHEMA,
    TABLE_CATALOG AS DATABASE,
    BYTES,
    TO_NUMBER(BYTES / POWER(1024, 3), 10, 2) AS GB,
    LAST_ALTERED AS LAST_USE,
    DATEDIFF('Day', LAST_USE, CURRENT_DATE) AS DAYS_SINCE_LAST_USE
FROM INFORMATION_SCHEMA.TABLES
WHERE DAYS_SINCE_LAST_USE > 90
ORDER BY BYTES DESC;

In [ ]:
-- Tables not used in ANY query in the last 90 days (based on Access History)
-- This is more accurate than LAST_ALTERED as it checks actual query access
WITH access_history AS (
    SELECT DISTINCT
        SPLIT(base.value:objectName, '.')[0]::STRING AS database_name,
        SPLIT(base.value:objectName, '.')[1]::STRING AS schema_name,
        SPLIT(base.value:objectName, '.')[2]::STRING AS table_name
    FROM snowflake.account_usage.access_history,
         LATERAL FLATTEN(base_objects_accessed) base
    WHERE query_start_time BETWEEN CURRENT_DATE() - 90 AND CURRENT_DATE()
)
SELECT
    tbl.table_catalog || '.' || tbl.table_schema || '.' || tbl.table_name AS fq_table_name
FROM
    snowflake.account_usage.tables tbl
    LEFT JOIN access_history ah
      ON tbl.table_name = ah.table_name
     AND tbl.table_schema = ah.schema_name
     AND tbl.table_catalog = ah.database_name
WHERE
    ah.table_name IS NULL
    AND tbl.deleted IS NULL;

---
## 5. Automatic Clustering

[Automatic Clustering](https://docs.snowflake.com/en/user-guide/tables-auto-reclustering) is a Snowflake-managed service that continuously re-orders data in clustered tables so that rows with similar clustering key values are co-located in the same micro-partitions.

### Why it matters
- Queries with predicates on the clustering key can **skip entire micro-partitions** that don't contain matching values (partition pruning).
- This reduces I/O, scanning time, and the warehouse size needed to run the query.

### How to evaluate clustering
- `SYSTEM$CLUSTERING_INFORMATION(table, key)` returns clustering depth and overlap statistics.
- Lower `average_depth` and `average_overlap` = better clustering.

### Demo below
We compare the same filter query (`l_shipdate BETWEEN ...`) on:
1. `lineitem` — ordered by `L_PARTKEY` (poor clustering on shipdate)
2. `lineitem_cl` — clustered by `LINEAR(l_shipdate)` (excellent clustering on shipdate)

Check the **Query Profile** after each execution to compare partitions scanned.

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE hol_compute_wh;
USE SCHEMA OPT_HOL.DEMO;

-- Check clustering information on the NON-clustered table (using l_shipdate as the hypothetical key)
SELECT SYSTEM$CLUSTERING_INFORMATION('LINEITEM', 'LINEAR(L_SHIPDATE)');

In [ ]:
-- Disable result cache so we see actual scan metrics in the Query Profile
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

-- Query the NON-clustered table with a date range filter
-- Check the Query Profile: expect MANY partitions scanned
SELECT *
FROM lineitem
WHERE l_shipdate BETWEEN '1995-01-01' AND '1995-03-01';

In [ ]:
-- Check clustering information on the CLUSTERED table
SELECT SYSTEM$CLUSTERING_INFORMATION('LINEITEM_CL');

In [ ]:
-- Query the CLUSTERED table with the same date range filter
-- Check the Query Profile: expect FAR FEWER partitions scanned
SELECT *
FROM lineitem_cl
WHERE l_shipdate BETWEEN '1995-01-01' AND '1995-03-01';

### Clustering Results

**Expected outcome:** The clustered table query scans significantly fewer micro-partitions, resulting in:
- Lower execution time
- Less data scanned (bytes)
- Better partition pruning ratio

Refer to [Clustering Considerations](https://docs.snowflake.com/en/user-guide/tables-clustering-keys#considerations-for-choosing-clustering-for-a-table) for guidance on choosing the right clustering key.

---
## 6. Materialized Views

A [Materialized View](https://docs.snowflake.com/en/user-guide/views-materialized) is a pre-computed dataset stored for later use. Key benefits:

- **Faster queries** — aggregation, projection, and selection are pre-computed
- **Automatic query rewrite** — Snowflake's optimizer can transparently use a materialized view even if the user query references the base table
- **Automatic maintenance** — Snowflake keeps the MV in sync with the base table (serverless background service)

### When to use Materialized Views
- Frequently executed, expensive aggregation queries
- Queries that consistently access a subset of columns
- Dashboards or reports with known, repeated query patterns

### Demo below
We query `lineitem_cl` with an aggregation that matches the `lineitem_mv` definition. Even though the query does NOT reference the MV directly, Snowflake's optimizer will rewrite the query to use it. Check the **Query Profile** to confirm.

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE hol_compute_wh;
USE SCHEMA OPT_HOL.DEMO;

-- This query does NOT reference lineitem_mv, but the optimizer may rewrite it
-- to use the MV since the aggregation pattern matches. Check Query Profile!
SELECT
    TO_CHAR(l_shipdate, 'YYYYMM') AS ship_month,
    l_orderkey,
    SUM(l_quantity * l_extendedprice) AS order_price,
    SUM(l_quantity * l_discount) AS order_discount
FROM
    lineitem_cl
WHERE
    l_orderkey BETWEEN 1000000 AND 2000000
GROUP BY
    ALL;

### Materialized View Outcome

In the Query Profile, you should see the optimizer scanning `LINEITEM_MV` instead of `LINEITEM_CL`. This demonstrates **automatic query rewrite** — the user doesn't need to know the MV exists.

Refer to [Materialized Views Best Practices](https://docs.snowflake.com/en/user-guide/views-materialized#best-practices-for-materialized-views) for considerations on when to use MVs vs. other optimization techniques.

---
## 7. Query Acceleration Service (QAS)

The [Query Acceleration Service](https://docs.snowflake.com/en/user-guide/query-acceleration-service) offloads portions of query processing to shared compute resources provided by Snowflake. It is particularly effective for:

- **Ad hoc analytics** with unpredictable data volumes
- **Outlier queries** that scan significantly more data than the average query on a warehouse
- **Large scans with selective filters** that benefit from parallel processing

### How it works
- When enabled on a warehouse, QAS detects queries that would benefit from additional parallelism.
- It offloads the scan/filter portion to elastic shared compute (no additional warehouse needed).
- The `query_acceleration_max_scale_factor` controls how much additional compute can be used (cost control).

### Cost considerations
- QAS is billed per-second for the additional compute consumed.
- Use `QUERY_ACCELERATION_ELIGIBLE` view to identify which queries benefit most before enabling.

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE hol_compute_wh;
USE SCHEMA OPT_HOL.DEMO;

-- Run this query BEFORE enabling QAS, then compare with AFTER enabling QAS
-- Check the Query Profile for execution time differences
SELECT
    i_brand,
    SUM(ss_quantity),
    SUM(ss_wholesale_cost),
    SUM(ss_sales_price),
    SUM(ss_list_price)
FROM
    snowflake_sample_data.tpcds_sf10tcl.store_sales ss
    JOIN snowflake_sample_data.tpcds_sf10tcl.item i
      ON i.i_item_sk = ss.ss_item_sk
WHERE
    ss_store_sk = 946
GROUP BY
    i_brand;

In [ ]:
-- Enable Query Acceleration on the warehouse with a max scale factor of 4
-- Scale factor limits how much additional compute QAS can use (cost guardrail)
ALTER WAREHOUSE hol_compute_wh
SET ENABLE_QUERY_ACCELERATION = TRUE
    QUERY_ACCELERATION_MAX_SCALE_FACTOR = 4;

In [ ]:
-- Find queries eligible for acceleration in the last 7 days
-- Use this to identify which warehouse workloads would benefit most from QAS
-- Replace the empty strings in WHERE clause with your warehouse name and user
SELECT
    LEFT(qh.QUERY_TEXT, 25) AS QueryCat,
    qh.USER_NAME,
    qae.WAREHOUSE_NAME,
    COUNT(*) AS QueryCount,
    AVG(qae.UPPER_LIMIT_SCALE_FACTOR) AS AvgScaleFactor,
    AVG(ELIGIBLE_QUERY_ACCELERATION_TIME) AS AvgTimeSavings,
    MAX(UPPER_LIMIT_SCALE_FACTOR) AS MaxScaleFactor,
    MIN(UPPER_LIMIT_SCALE_FACTOR) AS MinScaleFactor,
    SUM(ELIGIBLE_QUERY_ACCELERATION_TIME) AS TotalAccelerationTime
FROM
    SNOWFLAKE.ACCOUNT_USAGE.QUERY_ACCELERATION_ELIGIBLE qae
    JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh
      ON qh.query_id = qae.query_id
WHERE
    qae.WAREHOUSE_NAME IN ('HOL_COMPUTE_WH')  -- Replace with your warehouse
    AND ELIGIBLE_QUERY_ACCELERATION_TIME > 120
    AND qae.START_TIME >= CURRENT_DATE() - 7
GROUP BY
    ALL
ORDER BY
    TotalAccelerationTime DESC
LIMIT 1000;

In [ ]:
-- Estimate acceleration benefit for a specific query (replace with an actual query_id)
-- SELECT SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>');

---
## 8. Search Optimization Service

The [Search Optimization Service](https://docs.snowflake.com/en/user-guide/search-optimization-service) creates and maintains a persistent **search access path** that tracks which values exist in which micro-partitions. This enables partition skipping for:

- **Selective point lookups** (equality predicates on high-cardinality columns)
- **Substring and regex searches**
- **Semi-structured data queries** (VARIANT/OBJECT/ARRAY with equality, IN, ARRAY_CONTAINS)
- **Geospatial queries**

### How it works
- A background maintenance service builds the search access path.
- Queries transparently benefit — no query rewriting needed.
- Billed as a serverless service (storage + compute for maintenance).

### Cost estimation
Use `SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS()` to understand the expected cost BEFORE enabling the service on a table or specific columns.

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE hol_compute_wh;
USE SCHEMA OPT_HOL.DEMO;

-- Estimate cost of enabling Search Optimization on the entire CATALOG_RETURNS table
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('OPT_HOL.DEMO.CATALOG_RETURNS')
  AS estimate_for_full_table;

In [ ]:
-- Estimate cost for EQUALITY searches on a single column
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('OPT_HOL.DEMO.CATALOG_RETURNS', 'EQUALITY(CR_ITEM_SK)')
  AS estimate_single_column;

In [ ]:
-- Estimate cost for EQUALITY searches on multiple columns
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('OPT_HOL.DEMO.CATALOG_RETURNS', 'EQUALITY(CR_ITEM_SK, CR_RETURNED_DATE_SK)')
  AS estimate_multi_column;

In [ ]:
-- Estimate cost for SUBSTRING searches (useful for LIKE '%pattern%' queries)
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('OPT_HOL.DEMO.LINEITEM', 'SUBSTRING(L_SHIPMODE)')
  AS estimate_substring_search;

Refer to [Search Optimization Cost Estimation and Management](https://docs.snowflake.com/en/user-guide/search-optimization/cost-estimation) for detailed cost management guidance.

---
## 9. Cleanup (Optional)

Run the cell below to drop all objects created during this lab.

> **Warning:** This will permanently remove the `OPT_HOL` database and `hol_compute_wh` warehouse.

In [ ]:
-- Cleanup: drop all lab objects (uncomment to execute)
-- USE ROLE ACCOUNTADMIN;
-- DROP DATABASE IF EXISTS OPT_HOL;
-- DROP WAREHOUSE IF EXISTS hol_compute_wh;
-- DROP RESOURCE MONITOR IF EXISTS Credits_Quota_Monitoring;

---
## 10. Conclusion & Additional Resources

### What You Learned
- How to configure warehouse controls (auto-suspend, auto-resume, timeouts, resource monitors) to optimize credit usage
- How to use Account Usage views to identify cost and performance opportunities
- How to identify high-churn, short-lived, and unused tables for storage savings
- How to implement Automatic Clustering for faster filtered queries
- How Materialized Views provide automatic query rewrite for repeated aggregations
- How Query Acceleration Service helps outlier queries without upsizing the warehouse
- How Search Optimization Service accelerates point lookups and substring searches

### Further Reading
- [Definitive Guide to Managing Spend in Snowflake (PDF)](https://www.snowflake.com/wp-content/uploads/2023/10/Definitive-Guide-to-Managing-Spend-in-Snowflake.pdf)
- [Snowflake Education & Training](https://learn.snowflake.com/en/)
- [Snowflake Professional Services](https://www.snowflake.com/snowflake-professional-services/)